# SAC arrival_v2 — Tandem + SBS topology validation (vanilla SAC, single seed)

**前情**：commit `f179c5b` 已闭环 arrival_v2 reward 在两个 benchmark 上的 vanilla SAC 验证：
- `single_u10_cross_tgt15` (s0/10-D, U=1.0, λ=0.67, cross_stream): 5/5 gates PASS @ 1M
- `single_u15_upstream_tgt15` (s1/12-D, U=1.5, λ=1.0, upstream):     5/5 gates PASS @ 1.5M

**本 notebook 任务**：把 arrival_v2 验证扩展到双柱拓扑（同 U=1.5 / target=1.5 / λ=1.0 / upstream，仅换 topology）。两个 phase 物理机制不同：
- Phase 1 — `tandem_u15_upstream_tgt15`: 双柱串列 (G/D=3.5)，主导现象 = co-shedding 长尾涡街
- Phase 2 — `sbs_u15_upstream_tgt15`:    双柱并列 (G/D=3.5)，主导现象 = Coandă 偏转 + 不对称双尾

两个 phase **独立运行、独立 gate**（不互门控）。已验证的 P1 v6 (single upstream/U=1.5) 当作 topology baseline。

**配置矩阵**：

| Phase | benchmark | flow | obs | total_steps (起步) | 预算 |
|---|---|---|---|---:|---:|
| 1 | `tandem_u15_upstream_tgt15` | `wake_tandem_G35_v8_U1p50_Re250` | s1 / 12-D | 1.0M | ~2.5h L4 |
| 2 | `sbs_u15_upstream_tgt15`    | `wake_sbs_G35_v8_U1p50_Re250`    | s1 / 12-D | 1.0M | ~2.5h L4 |

**Gate**（每个 phase 独立判定，与 P1 v6 §8.3 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Single seed**：`seed=42`。如果 1.0M 收敛了就闭环；如果某一 phase eval 曲线尚在上升，把对应的 `TOTAL_STEPS` 改大（如 1.5M）后重跑同一 cell — `--resume` 会自动从 `trainer_state.json` 续训。

**输出根**：
- Phase 1: `experiments/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42/`
- Phase 2: `experiments/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42/`

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout）。

## 0. GPU sanity

In [1]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

Fri May  8 07:29:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
torch=2.10.0+cu128  cuda=True  device=NVIDIA L4


## 1. Mount Drive + cwd

和 commit `f179c5b` 同一个 working clone（含 arrival_v2 实现 + train_sac resume bug 修复）。

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. 通用配置 — Phase 1 (tandem) + Phase 2 (sbs)

唯一变量是 topology（`flow_path` + `benchmark_key`）。所有 SAC / env / reward 超参与 P1 v6 完全一致，便于和 single upstream 对照。

若某 phase 1.0M 不够，把对应 `TOTAL_STEPS` 改成 `1_500_000` 或 `2_000_000` 后重跑该 phase 的 train cell — skip/resume 逻辑会自动续训。

In [3]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== 全局 SAC / env config (两个 phase 共用) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's1'
HISTORY_LENGTH = 4
TASK_GEOMETRY = 'upstream'
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# ==== Phase 1 — tandem ====
T_BENCHMARK_KEY = 'tandem_u15_upstream_tgt15'
T_FLOW_PATH = 'wake_data/wake_tandem_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
T_TOTAL_STEPS = 1_000_000           # 起步 1M;不够就改大重跑此 cell
T_RUN_ROOT = Path('experiments/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
T_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
T_MANIFEST_PATH = Path(f'benchmarks/{T_BENCHMARK_KEY}.json')
T_RUN_ROOT_STR = str(T_RUN_ROOT)
T_CKPT_ROOT_STR = str(T_CKPT_ROOT)
T_MANIFEST_PATH_STR = str(T_MANIFEST_PATH)

# ==== Phase 2 — side-by-side ====
S_BENCHMARK_KEY = 'sbs_u15_upstream_tgt15'
S_FLOW_PATH = 'wake_data/wake_sbs_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
S_TOTAL_STEPS = 1_000_000           # 起步 1M
S_RUN_ROOT = Path('experiments/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
S_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')
S_MANIFEST_PATH = Path(f'benchmarks/{S_BENCHMARK_KEY}.json')
S_RUN_ROOT_STR = str(S_RUN_ROOT)
S_CKPT_ROOT_STR = str(S_CKPT_ROOT)
S_MANIFEST_PATH_STR = str(S_MANIFEST_PATH)

os.environ['PYTHONUNBUFFERED'] = '1'

print('---- Phase 1 (tandem) ----')
print(f'  benchmark    = {T_BENCHMARK_KEY}')
print(f'  flow         = {T_FLOW_PATH}')
print(f'  probe        = {PROBE_LAYOUT} / {TASK_GEOMETRY} / target={TARGET_SPEED}')
print(f'  total_steps  = {T_TOTAL_STEPS:,} (initial; bump and re-run if not converged)')
print(f'  run_root     = {T_RUN_ROOT}')
print(f'  ckpt_root    = {T_CKPT_ROOT}')
print()
print('---- Phase 2 (sbs) ----')
print(f'  benchmark    = {S_BENCHMARK_KEY}')
print(f'  flow         = {S_FLOW_PATH}')
print(f'  probe        = {PROBE_LAYOUT} / {TASK_GEOMETRY} / target={TARGET_SPEED}')
print(f'  total_steps  = {S_TOTAL_STEPS:,} (initial)')
print(f'  run_root     = {S_RUN_ROOT}')
print(f'  ckpt_root    = {S_CKPT_ROOT}')
print()
print(f'  seed={SEED}  num_envs={NUM_ENVS}  eval_every={EVAL_EVERY:,}  eval_episodes={EVAL_EPISODES}')

---- Phase 1 (tandem) ----
  benchmark    = tandem_u15_upstream_tgt15
  flow         = wake_data/wake_tandem_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy
  probe        = s1 / upstream / target=1.5
  total_steps  = 1,000,000 (initial; bump and re-run if not converged)
  run_root     = experiments/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42
  ckpt_root    = checkpoints/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42

---- Phase 2 (sbs) ----
  benchmark    = sbs_u15_upstream_tgt15
  flow         = wake_data/wake_sbs_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy
  probe        = s1 / upstream / target=1.5
  total_steps  = 1,000,000 (initial)
  run_root     = experiments/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42
  ckpt_root    = checkpoints/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42

  seed=42  num_envs=6  eval_every=25,00

## 3. Preflight — flow files / arrival_v2 candidate gate / reward unit tests / manifests

停止条件：
- 任一 phase flow 缺失 → raise（用户已确认 Drive 有；此处兜底）
- `validate_arrival_v2_candidate` 失败 → reward 公式不再满足 discounted unsafe-shortcut / terminal dominance
- `test_reward_objective.py` 失败 → reward 实现退化
- 任一 manifest 生成失败 → eval 不可重复

In [4]:
for label, fp in (('tandem', T_FLOW_PATH), ('sbs', S_FLOW_PATH)):
    p = Path(fp)
    if not p.exists():
        raise FileNotFoundError(f'missing {label} flow file: {p}')
    print(f'[OK] {label} flow: {p} ({p.stat().st_size / 1e6:.1f} MB)')

[OK] tandem flow: wake_data/wake_tandem_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy (440.6 MB)
[OK] sbs flow: wake_data/wake_sbs_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy (794.9 MB)


In [5]:
!python -u -m scripts.validate_arrival_v2_candidate


[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.


In [6]:
!python -u -m pytest tests/test_reward_objective.py -q

...............                                                          [100%]
15 passed in 17.56s


In [7]:
for key, mp in ((T_BENCHMARK_KEY, T_MANIFEST_PATH), (S_BENCHMARK_KEY, S_MANIFEST_PATH)):
    if not mp.exists():
        !python -u -m scripts.generate_standard_benchmarks --benchmarks {key} --episodes {EVAL_EPISODES}
    if not mp.exists():
        raise FileNotFoundError(f'manifest not generated: {mp}')
    print(f'[OK] manifest ready: {mp}')

[OK] manifest ready: benchmarks/tandem_u15_upstream_tgt15.json
[OK] manifest ready: benchmarks/sbs_u15_upstream_tgt15.json


## 4. Phase 1 — Tandem train (1.0M, fresh start with skip/resume)

Skip/resume 逻辑：
- `env_step >= T_TOTAL_STEPS` → skip（重跑此 cell 不会重训；要继续训需把 `T_TOTAL_STEPS` 改大后重跑此 cell + 上面 config cell）
- `0 < env_step < T_TOTAL_STEPS` → 用 `--resume` 续训（中断重连）
- `env_step == 0` → fresh start

训练时长（L4 / num_envs=6 / 1M steps）≈ 2.5h。Colab Pro+ 5h session 应可吃下两个 phase。

In [ ]:
t_state_path = T_RUN_ROOT / 'trainer_state.json'
if t_state_path.exists():
    t_state = json.loads(t_state_path.read_text(encoding='utf-8'))
    t_current_step = int(t_state.get('env_step', 0))
else:
    t_current_step = 0
print(f'[state] tandem env_step = {t_current_step:,} / target {T_TOTAL_STEPS:,}')

if t_current_step >= T_TOTAL_STEPS:
    print(f'[skip] tandem already trained to {t_current_step:,} >= {T_TOTAL_STEPS:,}')
    print('       要继续训:把 T_TOTAL_STEPS 改大,重跑 config cell + 此 cell')
elif t_current_step > 0:
    print(f'[resume] tandem continuing from {t_current_step:,} -> {T_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {T_RUN_ROOT_STR} \
        --total-steps {T_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {T_MANIFEST_PATH_STR} \
        --device {DEVICE}
else:
    print(f'[train] tandem fresh start -> {T_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {T_FLOW_PATH} \
        --task-geometry {TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {T_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {T_MANIFEST_PATH_STR} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {T_RUN_ROOT_STR} \
        --checkpoint-dir {T_CKPT_ROOT_STR}

[state] tandem env_step = 0 / target 1,000,000
[train] tandem fresh start -> 1,000,000
[train] episode=5 step=1416 return=-311.80 success=False time=118.0s geometry=upstream history=4
[train] episode=10 step=2808 return=-345.21 success=False time=42.1s geometry=upstream history=4
[train] episode=15 step=3336 return=-343.44 success=False time=92.4s geometry=upstream history=4
[train] episode=20 step=4866 return=-281.90 success=False time=28.7s geometry=upstream history=4
[train] episode=25 step=5736 return=-284.53 success=False time=20.6s geometry=upstream history=4 | q1=281.587 actor=0.986 alpha=0.193
[train] episode=30 step=7152 return=-377.98 success=False time=190.0s geometry=upstream history=4 | q1=538.862 actor=2.094 alpha=0.181
[train] episode=35 step=8358 return=-277.05 success=False time=9.6s geometry=upstream history=4 | q1=267.788 actor=3.952 alpha=0.170
[train] episode=40 step=9204 return=-316.85 success=False time=58.1s geometry=upstream history=4 | q1=486.654 actor=4.474 a

## 5. Phase 1 — Tandem summary + gate

Gate 5 条与 P1 v6 同口径。Gate summary 落到 `results/tandem_validation_gate_summary.json`。

In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s1 / upstream / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

tandem_summary = summarize_phase(
    T_RUN_ROOT,
    T_TOTAL_STEPS,
    'TANDEM_VALIDATION',
    'tandem_validation_gate_summary.json',
)
TANDEM_PASS = tandem_summary['all_pass']
print()
print(f'TANDEM_PASS = {TANDEM_PASS}')

TANDEM_VALIDATION  (arrival_v2 / s1 / upstream / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 1.0000   gate >= 0.85
  peak_success_rate     : 1.0000   @ 475,002
  last100k_mean_success : 0.9750   gate >= 0.9000
  final_oob_rate        : 0.0000   gate <= 0.10
  obs_dim               : 56
  context_obs           : True
  timeout_bootstrap     : terminal
  termination           : {'goal': 30}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           1.000000   138.180850          2.962520   119.920000             0.932173
   625002           1.000000   132.507112          5.820720   117.913333             0.932200
   650004           1.000000   132.498663          5.986779   102.556667             0.932257
   675000           0.966667   126.291750          6.179022   103.693333             0.925076
   700002        

## 6. Phase 2 — SBS train (1.0M, independent)

**独立运行**：sbs 不门控于 tandem。两种拓扑物理机制不同（co-shedding vs Coandă jet），都值得独立看一看。

Skip/resume 逻辑同 tandem。

In [ ]:
s_state_path = S_RUN_ROOT / 'trainer_state.json'
if s_state_path.exists():
    s_state = json.loads(s_state_path.read_text(encoding='utf-8'))
    s_current_step = int(s_state.get('env_step', 0))
else:
    s_current_step = 0
print(f'[state] sbs env_step = {s_current_step:,} / target {S_TOTAL_STEPS:,}')

if s_current_step >= S_TOTAL_STEPS:
    print(f'[skip] sbs already trained to {s_current_step:,} >= {S_TOTAL_STEPS:,}')
    print('       要继续训:把 S_TOTAL_STEPS 改大,重跑 config cell + 此 cell')
elif s_current_step > 0:
    print(f'[resume] sbs continuing from {s_current_step:,} -> {S_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {S_RUN_ROOT_STR} \
        --total-steps {S_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {S_MANIFEST_PATH_STR} \
        --device {DEVICE}
else:
    print(f'[train] sbs fresh start -> {S_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {S_FLOW_PATH} \
        --task-geometry {TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {S_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {S_MANIFEST_PATH_STR} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {S_RUN_ROOT_STR} \
        --checkpoint-dir {S_CKPT_ROOT_STR}

[state] sbs env_step = 0 / target 1,000,000
[train] sbs fresh start -> 1,000,000
[train] episode=5 step=870 return=-352.81 success=False time=41.4s geometry=upstream history=4
[train] episode=10 step=1746 return=-405.33 success=False time=106.6s geometry=upstream history=4
[train] episode=15 step=2892 return=-346.63 success=False time=45.1s geometry=upstream history=4
[train] episode=20 step=3558 return=-357.45 success=False time=35.7s geometry=upstream history=4
[train] episode=25 step=4746 return=-339.74 success=False time=129.4s geometry=upstream history=4
[train] episode=30 step=5562 return=-343.31 success=False time=109.3s geometry=upstream history=4 | q1=419.736 actor=1.687 alpha=0.195
[train] episode=35 step=7128 return=-405.57 success=False time=149.4s geometry=upstream history=4 | q1=323.256 actor=2.420 alpha=0.181
[train] episode=40 step=8088 return=-276.14 success=False time=21.4s geometry=upstream history=4 | q1=289.238 actor=3.624 alpha=0.173
[train] episode=45 step=9768 r

## 7. Phase 2 — SBS summary + gate

In [ ]:
sbs_summary = summarize_phase(
    S_RUN_ROOT,
    S_TOTAL_STEPS,
    'SBS_VALIDATION',
    'sbs_validation_gate_summary.json',
)
SBS_PASS = sbs_summary['all_pass']
print()
print(f'SBS_PASS = {SBS_PASS}')

SBS_VALIDATION  (arrival_v2 / s1 / upstream / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 1.0000   gate >= 0.85
  peak_success_rate     : 1.0000   @ 625,002
  last100k_mean_success : 0.9833   gate >= 0.9000
  final_oob_rate        : 0.0000   gate <= 0.10
  obs_dim               : 56
  context_obs           : True
  timeout_bootstrap     : terminal
  termination           : {'goal': 30}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           0.966667   123.860236          5.503943   152.616667             0.931958
   625002           1.000000   141.515199          1.244344   136.600000             0.937064
   650004           0.900000   126.371848          0.686876   150.780000             0.929746
   675000           0.866667   120.580633          1.104732   139.433333             0.929093
   700002           

## 8. Combined verdict + 如何延长训练

总结两个 phase 的 5/5 gate 状态。两个 phase 互不门控；任一 phase fail，单独看其 last100k_mean / peak / OOB 哪条 fail，再决定延长 budget 还是改 reward。

**延长训练的标准流程**（不需要重跑 preflight）：

1. 在 §2 config cell 把对应的 `T_TOTAL_STEPS` 或 `S_TOTAL_STEPS` 改大（如 `1_500_000`）
2. 重跑该 phase 的 config cell（让新值生效）
3. 重跑该 phase 的 train cell — `--resume` 自动续训
4. 重跑该 phase 的 summary cell — gate summary JSON 会被覆盖

In [ ]:
print('=' * 96)
print('TOPOLOGY VALIDATION — combined verdict')
print('-' * 96)
print(f"  TANDEM_PASS = {TANDEM_PASS}")
print(f"    final={tandem_summary['final_success_rate']:.4f}  "
      f"peak={tandem_summary['peak_success_rate']:.4f} @ {tandem_summary['peak_step']:,}  "
      f"last100k_mean={tandem_summary['last100k_mean_success']:.4f}  "
      f"oob={tandem_summary['final_oob_rate']:.4f}")
print(f"  SBS_PASS    = {SBS_PASS}")
print(f"    final={sbs_summary['final_success_rate']:.4f}  "
      f"peak={sbs_summary['peak_success_rate']:.4f} @ {sbs_summary['peak_step']:,}  "
      f"last100k_mean={sbs_summary['last100k_mean_success']:.4f}  "
      f"oob={sbs_summary['final_oob_rate']:.4f}")
print('=' * 96)

combined = {
    'experiment': 'arrival_v2_topology_validation',
    'commit_baseline': 'f179c5b',
    'seed': SEED,
    'tandem': tandem_summary,
    'sbs': sbs_summary,
    'tandem_pass': TANDEM_PASS,
    'sbs_pass': SBS_PASS,
    'both_pass': bool(TANDEM_PASS and SBS_PASS),
}
out_dir = Path('experiments/arrival_v2_prototype/topology_validation_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(combined, indent=2), encoding='utf-8')
print(f'[saved] {out_path}')

TOPOLOGY VALIDATION — combined verdict
------------------------------------------------------------------------------------------------
  TANDEM_PASS = True
    final=1.0000  peak=1.0000 @ 475,002  last100k_mean=0.9750  oob=0.0000
  SBS_PASS    = True
    final=1.0000  peak=1.0000 @ 625,002  last100k_mean=0.9833  oob=0.0000
[saved] experiments/arrival_v2_prototype/topology_validation_summary/combined_gate_summary.json
